# Conjugate gradient for the 2D Poisson system (research prototype)

Solve `A x = b` where the unknowns live on an `m x m` grid (row-major, index
`idx = i*m + j`) and `A` is the matrix-free 5-point Poisson operator with Dirichlet
boundaries OUTSIDE the grid -- a neighbour off the edge contributes zero:

    (A v)[i,j] = 4*v[i,j] - v[i-1,j] - v[i+1,j] - v[i,j-1] - v[i,j+1].

The solver is standard conjugate gradient starting from `x = 0`, run for a FIXED
number of iterations (no convergence test, so the run is deterministic). The
right-hand side `b` uses a fixed deterministic fill. After `ITERS` iterations we
report the checksum (sum of all entries of `x`) as the oracle.

In [ ]:
M_TEST = 64
ITERS = 100

# deterministic right-hand side, one value per grid cell (idx = i*m + j)
def b_of(idx):
    return ((idx * 97) % 211) / 211.0

b = [b_of(idx) for idx in range(M_TEST * M_TEST)]

In [ ]:
# matrix-free 5-point Poisson operator: neighbours off the edge are zero (Dirichlet)
def apply_A(v, m):
    q = [0.0] * (m * m)
    for i in range(m):
        for j in range(m):
            idx = i * m + j
            s = 4.0 * v[idx]
            if i > 0:     s -= v[idx - m]
            if i < m - 1: s -= v[idx + m]
            if j > 0:     s -= v[idx - 1]
            if j < m - 1: s -= v[idx + 1]
            q[idx] = s
    return q

def dot(u, v):
    s = 0.0
    for a, c in zip(u, v):
        s += a * c
    return s

# standard conjugate gradient from x = 0, run for a FIXED number of iterations
def cgsolve(b, m, iters):
    n = m * m
    x = [0.0] * n
    r = b[:]
    p = r[:]
    rr = dot(r, r)
    for _ in range(iters):
        q = apply_A(p, m)
        alpha = rr / dot(p, q)
        for k in range(n):
            x[k] += alpha * p[k]
        for k in range(n):
            r[k] -= alpha * q[k]
        rr_new = dot(r, r)
        beta = rr_new / rr
        rr = rr_new
        for k in range(n):
            p[k] = r[k] + beta * p[k]
    return x

x = cgsolve(b, M_TEST, ITERS)
checksum = sum(x)
print('checksum =', checksum)   # -> 311942.8986866846

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any
# language) must reproduce this checksum within tol. The final solution entries
# are handed over as the terms of the checksum so the extraction tool can measure
# how much the sum legitimately moves under reordering. (That spread only covers
# the final reduction -- CG also amplifies the rounding of its dot products across
# all 100 iterations -- so it is a lower bound on how much correct ports may
# disagree, and tol must stay well above it.)
ORACLE = {'kernel': 'cg',
          'params': {'M': M_TEST, 'ITERS': ITERS},
          'expected': {'CHECKSUM': checksum},
          'terms': {'CHECKSUM': x},
          'tol': 1e-3}